# NoSQL Databases and MongoDB

## Introduction

Relational databases are excellent for structured, consistent data. But not all data fits neatly into rows and columns. **NoSQL** (Not Only SQL) databases were built for data that is variable in shape, nested, or needs to scale horizontally across many machines. This notebook covers why NoSQL exists, what a document store is, and how to use MongoDB from Python.

## Objectives

You will be able to:

- Describe situations where a NoSQL database is a better fit than a relational database
- Name the four main categories of NoSQL databases
- Connect to a MongoDB database using `pymongo`
- Perform CRUD operations: insert, find, update, and delete documents

---

## Why NoSQL?

Relational databases require a **fixed schema** — you define the columns upfront, and every row must conform. This works beautifully for structured business data (orders, invoices, HR records), but breaks down for:

- **Variable-length records** — a chat log with 2 messages and one with 2,000 messages cannot easily share the same table schema.
- **Nested or hierarchical data** — a product with an arbitrary number of sub-components.
- **Rapid iteration** — when the shape of data isn't known in advance and the schema needs to evolve.
- **Horizontal scale** — when data is too large for one machine and needs to be partitioned across many.

## Four Categories of NoSQL Databases

| Category | Stores | Examples | Common Use |
|----------|--------|----------|------------|
| **Document Store** | JSON-like documents | MongoDB, CouchDB | Web/mobile backends, logs |
| **Key-Value Store** | Simple value by key | Redis, DynamoDB | Caching, sessions |
| **Column Store** | Grouped columns | Cassandra, HBase | Time series, analytics at scale |
| **Graph Database** | Nodes and edges | Neo4j, Amazon Neptune | Social networks, fraud detection |

We'll focus on **Document Stores** — specifically MongoDB.

---

## Document Stores

A document store saves each record as a self-contained **document** — typically a JSON object. Unlike SQL tables:

- No fixed schema: each document can have different fields.
- Documents can be nested arbitrarily: a field's value can itself be another document or a list.
- Collections group related documents (like a SQL table), but impose no schema constraints.

Example MongoDB document:

```json
{
  "_id": ObjectId("5d35ffb254722712a2675be8"),
  "name": "John Doe",
  "address": "123 Elm Street",
  "age": 28,
  "orders": [
    {"item": "Widget", "qty": 3, "price": 9.99},
    {"item": "Gadget", "qty": 1, "price": 49.99}
  ]
}
```

Every document gets a unique `_id` assigned by the database — the MongoDB equivalent of a primary key.

---

## Connecting to MongoDB with `pymongo`

> **Setup:** MongoDB must be running locally. Install via [MongoDB Community Server](https://www.mongodb.com/try/download/community) or Docker: `docker run -d -p 27017:27017 mongo`

```python
import pymongo
client = pymongo.MongoClient('mongodb://127.0.0.1:27017/')
db = client['my_database']          # creates if it doesn't exist
collection = db['my_collection']    # creates if it doesn't exist
```

MongoDB uses **lazy creation** — the database and collection are not created on disk until the first document is inserted.

In [ ]:
import pymongo

client = pymongo.MongoClient('mongodb://127.0.0.1:27017/')
db = client['example_database']
collection = db['customers']

# Database doesn't appear until data is inserted
print('Databases before insert:', client.list_database_names())

---

## INSERT — Adding Documents

Use `.insert_one()` for a single document or `.insert_many()` for a list.

In [ ]:
# Insert a single document
result = collection.insert_one({'name': 'John Doe', 'address': '123 Elm Street', 'age': 28})
print('Inserted id:', result.inserted_id)

In [ ]:
# Insert multiple documents
more_customers = [
    {'name': 'Jane Doe',    'address': '234 Elm Street',  'age': 7},
    {'name': 'Santa Claus', 'address': 'The North Pole',  'age': 547},
    {'name': 'John Doe jr.','address': '',                'age': 0.5},
]
result2 = collection.insert_many(more_customers)
print('Inserted ids:', result2.inserted_ids)

In [ ]:
# Database now shows up
print('Databases after insert:', client.list_database_names())

---

## FIND — Querying Documents

`.find({})` returns all documents. Pass a filter dictionary to match specific field values. The second optional argument specifies which fields to include (value `1`) or exclude (value `0`).

In [ ]:
# All documents
for doc in collection.find({}):
    print(doc)

In [ ]:
# Filter by exact field value
for doc in collection.find({'name': 'Santa Claus'}):
    print(doc)

In [ ]:
# Select specific fields only (1 = include, 0 = exclude)
for doc in collection.find({}, {'_id': 0, 'name': 1, 'address': 1}):
    print(doc)

In [ ]:
# Comparison operators: $gt, $lt, $gte, $lte, $ne
for doc in collection.find({'age': {'$gt': 20}}):
    print(doc)

---

## UPDATE — Modifying Documents

`.update_one()` modifies the first matching document. Use the `$set` operator to change or add fields.

In [ ]:
# Update an existing field
collection.update_one({'name': 'John Doe'}, {'$set': {'age': 29}})

# Add a new field that didn't exist before
collection.update_one({'name': 'John Doe'}, {'$set': {'birthday': '02/20/1986'}})

# Verify
print(collection.find_one({'name': 'John Doe'}))

---

## DELETE — Removing Documents

`.delete_one()` removes the first matching document. `.delete_many({})` with an empty filter removes **all** documents — use with care.

In [ ]:
# Delete one by name
result = collection.delete_one({'name': 'John Doe'})
print(f'Deleted {result.deleted_count} document(s)')

# Delete all under age 10 using a modifier
result2 = collection.delete_many({'age': {'$lt': 10}})
print(f'Deleted {result2.deleted_count} document(s)')

# What's left?
for doc in collection.find({}):
    print(doc)

In [ ]:
# Clean up — delete everything from the collection
collection.delete_many({})

---

## Practice

Work with a new collection called `'lab_customers'` inside `'lab_db'`.

| Name | Email | Address | Balance | Notes |
|------|-------|---------|---------|-------|
| John Smith | j.smith@example.com | 123 Mulberry Lane | 0.0 | Called support, issue unresolved |
| Jane Smith | jane.smith@example.com | *(omit — null)* | 25.00 | *(omit)* |
| Adam Enbar | adam@school.com | 11 Broadway | 14.99 | Recurring billing |
| Avi Flombaum | avi@school.com | 11 Broadway | 0.0 | *(omit)* |
| Steven S. | steven@gmail.net | *(omit — null)* | -20.23 | Refunded for overpayment |

In [ ]:
lab_db = client['lab_db']
lab_col = lab_db['lab_customers']

In [ ]:
# Create the 5 customer documents (omit null fields entirely) and insert_many()
# Note: balance can be negative — store as REAL / float


In [ ]:
# Confirm: print the inserted_ids


In [ ]:
# Query 1: return name and email for every customer (exclude _id)


In [ ]:
# Query 2: get the record for 'John Smith' by name


In [ ]:
# Query 3: names, emails, and balances for customers with a positive balance


In [ ]:
# Update: set John Smith's mailing address to '367 55th St., apt 2A'


In [ ]:
# Add birthdays to all 5 customers using the mapping below
# Write a loop or function — avoid repeating update_one() 5 times
birthdays = {
    'John Smith':   '02/20/1986',
    'Jane Smith':   '07/07/1983',
    'Adam Enbar':   '12/02/1982',
    'Avi Flombaum': '04/17/1983',
    'Steven S.':    '08/30/1991',
}


In [ ]:
# Verify all records have a birthday field
for doc in lab_col.find({}, {'_id': 0, 'name': 1, 'birthday': 1}):
    print(doc)

---

## Summary

In this notebook you learned:

- When to choose NoSQL over a relational database (variable schema, nested data, horizontal scale)
- The four NoSQL categories: document stores, key-value stores, column stores, graph databases
- How MongoDB stores records as schema-free JSON documents in collections
- `pymongo` CRUD: `.insert_one()` / `.insert_many()`, `.find()` with filters and field selectors, `$gt`/`$lt` modifiers, `.update_one()` with `$set`, `.delete_one()` / `.delete_many()`

Next: [02 — Vector Databases](02_vector_databases.ipynb)